In [50]:
from langgraph.graph import StateGraph , START , END
from langchain_mistralai import ChatMistralAI
from typing import TypedDict
from rich import print

In [51]:
class BatsmanState(TypedDict):
    
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    balls_per_boundary: float
    boundary_percent: float
    summary: str



In [52]:
def calc_strike_rate(state : BatsmanState):
    
    runs = state['runs']
    balls = state['balls']

    strike_rate = (runs / balls) * 100

    # state['strike_rate'] = strike_rate

    return {"strike_rate" : strike_rate}

In [53]:
def calc_balls_per_boundary(state: BatsmanState):
    
    balls = state['balls']
    fours = state['fours']
    sixes = state['sixes']

    balls_per_boundary = balls/(fours + sixes)

    # state['balls_per_boundary'] = balls_per_boundary

    return {'balls_per_boundary' : balls_per_boundary}

In [54]:
def calc_boundary_percent(state : BatsmanState) -> float:
    
    balls = state['balls']
    fours = state['fours']
    sixes = state['sixes']

    boundary_percent = ((fours + sixes)/balls) * 100
    # state['boundary_percent'] = boundary_percent
    
    return {'boundary_percent': boundary_percent}

In [55]:
def summary(state : BatsmanState):
    summary_text = f"""
Strike Rate - {state['strike_rate']}
Balls per Boundary - {state['balls_per_boundary']}
Boundary Percent - {state['boundary_percent']}
"""

    return {'summary' : summary_text}


In [56]:
graph = StateGraph(BatsmanState)

graph.add_node('calc_strike_rate', calc_strike_rate)
graph.add_node('calc_balls_per_boundary', calc_balls_per_boundary)
graph.add_node('calc_boundary_percent', calc_boundary_percent)
graph.add_node('summary', summary)


graph.add_edge(START , 'calc_strike_rate')
graph.add_edge(START , 'calc_balls_per_boundary')
graph.add_edge(START , 'calc_boundary_percent')
graph.add_edge('calc_strike_rate' , 'summary')
graph.add_edge('calc_balls_per_boundary' , 'summary')
graph.add_edge('calc_boundary_percent' , 'summary')
graph.add_edge('summary' ,  END)

# compile the graph
workflow = graph.compile()

print(workflow)

# execute the graph
workflow.invoke({'runs' : 80 , 'balls' : 24 , 'fours' : 5 , 'sixes' : 8})

<langgraph.graph.state.CompiledStateGraph object at 0x0000026B1062B9B0>

{'runs': 80,
 'balls': 24,
 'fours': 5,
 'sixes': 8,
 'strike_rate': 333.33333333333337,
 'balls_per_boundary': 1.8461538461538463,
 'boundary_percent': 54.166666666666664,
 'summary': '\nStrike Rate - 333.33333333333337\nBalls per Boundary - 1.8461538461538463\nBoundary Percent - 54.166666666666664\n'}